<div style="background-color: #000000; padding: 10px; border-radius: 5px;">
<img src="images/logo.png" width="100">

In [1]:
# find the center of the ligand
import MDAnalysis as mda
original_structure = mda.Universe("protein_structures/chain_A.pdb")
ligand_mda = original_structure.select_atoms("resname CP6 and resid 609")
# Get the center of the ligand as the "pocket center"
pocket_center = ligand_mda.center_of_geometry()## Fill in code for measuring the center of geometry of the ligand.
print(pocket_center)

[28.16464705  5.31058823 58.67170626]


In [2]:
from vina import Vina
v = Vina(sf_name="vina")

In [3]:
#! grep " A  86 " protein_structures/chain_A_cleanH.pdb

In [4]:
!mk_prepare_receptor.py \
-i protein_structures/chain_A_cleanH.pdb \
-o protein_structures/chain_A_clean \
 -p

@> 3707 atoms and 1 coordinate set(s) were parsed in 0.03s.

Files written:
protein_structures/chain_A_clean.pdbqt <-- static (i.e., rigid) receptor input file


In [7]:
from vina import Vina
from pathlib import Path
import os
ligand_dir = Path("pdbqt")
out_dir = Path("docking results")
out_dir.mkdir(exist_ok=True)
for ligand in ligand_dir.glob("*.pdbqt"):
    v = Vina(sf_name="vina", seed=0)
    v.set_receptor("protein_structures/chain_A_clean.pdbqt")
    v.set_ligand_from_file(str(ligand))
    v.compute_vina_maps(
        center=pocket_center,
        box_size=[20, 20, 20]
    )
    v.dock(exhaustiveness=15, n_poses=10)
    print(f"\n{ligand.name}")
    for i, e in enumerate(v.energies(), 1):
        print(f"Pose {i}: {e[0]:.2f} kcal/mol")
    v.write_poses(
        str(out_dir / f"{ligand.stem}_poses.pdbqt"),
        n_poses=5,
        overwrite=True
    )


mode |   affinity | dist from best mode
     | (kcal/mol) | rmsd l.b.| rmsd u.b.
-----+------------+----------+----------
   1       -7.003          0          0
   2       -6.668      3.124      4.461
   3       -6.652      3.153      4.349
   4       -6.602      3.666      5.654
   5       -6.478      3.101      4.404
   6       -6.415      3.325      4.788
   7       -6.409      4.277      6.449
   8       -6.312      4.665      6.107
   9       -6.283      2.951      4.866
  10       -6.273      3.168      4.819
Computing Vina grid ... done.
Performing docking (random seed: 602986555) ... 
0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************

pyrimethamine.pdbqt
Pose 1: -7.55 kcal/mol
Pose 2: -6.81 kcal/mol
Pose 3: -6.76 kcal/mol
Pose 4: -6.68 kcal/mol
Pose 5: -6.59 kcal/mol
Pose 6: -6.50 kcal/mol
Pose 7: -6.49 kcal/mol
Pose 8: -6.48 kcal/mol
Pose 9: -6.37 kcal/mol

mode |   a

In [6]:
#convert pdbqt dock poses into SDF
import glob
import os

for pdbqt in glob.glob("docking results/*.pdbqt"):
    sdf = os.path.splitext(pdbqt)[0] + ".sdf"
    !mk_export.py "$pdbqt" -s "$sdf"